In [31]:
import numpy as np
import math
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
from scipy.optimize import fsolve, differential_evolution,minimize


In [32]:
def bound_on_Dx(m1, Omega,Tau):
    return m1 * np.exp(-np.power(m1,1-Tau)/ (3 * math.log(m1) ** Omega))

def bound_on_Fx(value, Zeta, Log_space=False):
    if Log_space:
        M1 = m_func(value, Zeta, Log_space)
        return M1 * np.exp(-np.exp(value * (1 - Zeta)) / 4)
    else:
        return m_func(value, Zeta) * np.exp(-np.power(value, 1 - Zeta) / 4)

def bound_on_Tx(m1, K):
    return np.power(m1, -K / 6 + 2)

In [33]:
def zeta_bound(Epsilon):
    return 1 / (1 - Epsilon) * 3 / 4


def epsilon_bound(Zeta):
    return 1 - 3 / (4 * Zeta)

def optimal_epsilon(Zeta):
    return 1 / 2 - 3 / (8 * Zeta)


In [34]:
def binary_entropy(p):
    if p <= 0 or p >= 1:
        return 0.0
    # math.log2 works on floats; if using arrays, use math.log2
    return -(p * math.log2(p) + (1 - p) * math.log2(1 - p))

def m_func(value,Zeta, Log_space=False):
    if Log_space:
        return np.ceil(np.exp(value*Zeta))
    else:
        return np.ceil(np.power(value,Zeta))

def log_binom_bound(value,Epsilon):
    cst = 2*binary_entropy(Epsilon)*math.log(2)
    return value*cst

def C_epsilon(Epsilon,c0): #2H(˜ε) log 2 + ˜ε log ˜ε − ˜ε
    Tilda_epsilon = Epsilon*c0
    return 2*binary_entropy(Tilda_epsilon)*math.log(2) + Tilda_epsilon*math.log(Tilda_epsilon) - Tilda_epsilon

def exps_bound_of_Ee(value,Epsilon,Delta,Zeta,c0):
    Epsilon_tilda = Epsilon*c0
    L_term = 2*Epsilon+Epsilon_tilda + 4*Zeta*(Delta-Epsilon) -2*Delta
    #linear term is 2ε + ˜ε + 4ζ(δ − ε) -2δ
    constant_term = (2*Epsilon*math.log(3*Delta)+C_epsilon(Epsilon,c0)+2*Delta - 2*Delta*math.log(Delta)  + 4*(Delta-Epsilon)*math.log(c0) )

    # print(f"The constant term of Event E is {constant_term:.4e}, while the L_term is {L_term:.4e}")

    exp_term = -value/2 - math.log(2*np.pi*Epsilon_tilda)/2-math.log(1-Epsilon_tilda)

    if L_term >=0:
        return False

    result = constant_term + value*L_term + np.exp(-value)*exp_term
    #this result is then multiplied by n, and exponentiated, so value becomes negligible
    return result



In [35]:
def A_condition(value,Zeta,Epsilon,Gamma,Omega,Tau,K,c0):
    C_k = 2*np.exp(2)*(4*K+1)*(2*K+1) + 2*np.exp(2)
    # C_k = 2*np.exp(2)*(4*K)*(2*K)
    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda = Epsilon*c0
    log_M = math.log(M1)
    Term1 = Gamma/(C_k*np.power(M1,2*Tau)*np.power(log_M,2+2*Omega)) * np.power(1-1/log_M**Omega,4*K*log_M)
    #coupled with e^L
    Term2 = C_epsilon(Epsilon,c0)
    #constant
    Term3 = -(value/2 + math.log(2*math.pi*Epsilon_tilda)/2 + math.log(1-Epsilon_tilda) )#+ Zeta*value*(2-K/6))
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return -Term1*np.exp(value) +Term2+Term3*np.exp(-value)+ Term4*value

def H_condition(value,Zeta,Gamma,Epsilon,Tau,c0,Constant1=3/(52*np.exp(2))):#constant 1 is the value that arises from lemma 7.2

    M1 = m_func(value,Zeta,Log_space=True)
    Epsilon_tilda=Epsilon*c0
    Term1 = Constant1*Gamma/np.power(M1,2*Tau)
    #coupled with e^L
    Term2 = C_epsilon(Epsilon,c0)
    #constant
    Term3 = -(value/2 + math.log(2*math.pi*Epsilon_tilda)/2+math.log(1-Epsilon_tilda))
    #coupled with e^-L
    Term4 = Epsilon_tilda
    #coupled with L
    return -Term1*np.exp(value) +Term2+Term3*np.exp(-value)+Term4*value


In [36]:
def gamma_n(value,Zeta,Delta,Omega,Tau,Log_space=True):
    if Log_space:
        n_value = np.exp(value)
        M1 = m_func(value,Zeta,Log_space=Log_space)
        return Delta**2/4 - Delta/(4*n_value) - 16*np.power(M1,1-2*Tau)/np.power(math.log(M1),2*Omega)

def thm31_req(value,Zeta,Delta, Omega, Tau, bound,Log_space=True):
    if Log_space:
        M1 = m_func(value,Zeta,Log_space=Log_space)
        return bound > 4/M1 + 16*np.power(M1,1-2*Tau)/np.power(math.log(M1),2*Omega) + Delta/(4*np.exp(value))

def lemma42_req(value,Zeta,Epsilon,Log_space=True):
    M1 = m_func(value, Zeta, Log_space)
    if Log_space:
        return 2 + M1  <= Epsilon**2*np.exp(value)

In [37]:
# def least_L_with_params(zeta, delta, gamma, omega, k, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):
def least_L_with_params(zeta, delta, omega, k, tau=1/2, epsilon=0.25, c0=1+1e-30, tol=1e-9):
    # Assuming epsilon and zeta_bound are defined
    params = {
        "zeta": zeta,
        "delta": delta,
        # "gamma": gamma,
        "k": k
    }

    # Define requirements as (Current Value, Boolean Condition, Description)
    requirements = {
        "Tau": (tau,1>tau>=0.5, f"0.5 <= τ < 1" ),
        "Zeta": (zeta, 1 > zeta > 2/(2+tau), f"2/(2+τ) < ζ < 1"),
        "Delta": (delta, delta < epsilon*(1 - c0/(4*zeta-2)), f" δ < {epsilon*epsilon*(1 - (2+c0)/(4*zeta)):.3e}" ),
        # "Gamma": (gamma, 0 < gamma < delta ** 2 / 4, f"0 < γ < {delta**2 / 4:.2e}"),
        "k": (k, k >12 , "k > 12")
    }

    # Check for failures
    failed = [name for name, (val, met, desc) in requirements.items() if not met]

    if failed:
        # print("═══ PARAMETER CHECK FAILED ═══")
        # for name, (val, met, desc) in requirements.items():
        #     status = "✓" if met else "✗"
        #     print(f"{status} {name:6} : {val:<8} | Required: {desc}")

        return 1e12
        # raise ValueError(f"Constraint violation in: {', '.join(failed)}")

    # print("✓ All parameters satisfy requirements.")

    # Define a helper to check all conditions for a given L
    def is_feasible(L):
        gamma_opt = gamma_n(value=L,Zeta=zeta,Delta=delta,Omega=omega,Tau=tau,Log_space=True)
        H_exp = H_condition(value=L, Zeta=zeta, Gamma=gamma_opt,Tau=tau, Epsilon=epsilon,c0=c0)
        A_exp = A_condition(value=L, Zeta=zeta, Epsilon=epsilon, Gamma=gamma_opt, K=k, Tau=tau, Omega=omega,c0=c0)

        threshold = -1e-50 # requires all the different exponent/n to be less than this number, because when multiplied by n, this becomes negligible
                # Calculate bounds
        Fx_bound = bound_on_Fx(value=L, Zeta=zeta, Log_space=True)
        M = m_func(value=L, Zeta=zeta, Log_space=True)
        Dx_bound = bound_on_Dx(M, omega,tau)
        Tx_bound = bound_on_Tx(M, k)

        Ee_bound = exps_bound_of_Ee(value=L,Epsilon=epsilon,Delta=delta,Zeta=zeta,c0=c0)
        # print(f"Bound on event E = {Ee_bound:.2f}")
        conds = [
            # thm31_req(L, Zeta=zeta, Delta=delta, Omega=omega, Tau=tau,bound=delta**2/4-gamma),
            # lemma42_req(L,Zeta=zeta,Epsilon=epsilon,Log_space=True),
            Ee_bound<threshold,
            np.exp(L)*delta > M, #ensures \delta*n > m
            H_exp < threshold,
            A_exp < threshold
        ]
        # print(conds)
        return 2*(Fx_bound+Dx_bound+Tx_bound) <1 and all(conds)
    # Bisection Search
    low = 200
    high = 700

    # First, check if high is even feasible
    if not is_feasible(high):
        return 1e12

    best_L = high
    while (high - low) > tol:
        mid = (low + high) / 2
        if is_feasible(mid):
            best_L = mid
            high = mid  # Try to find a smaller L in the lower half
        else:
            low = mid   # Need a larger L, look in the upper half

    # print(f"logn is around {best_L:.10f} and so n is around {np.exp(best_L):.8e}")
    return best_L

In [38]:
ZETA= 8.00000000000001043610e-01
DELTA = 1.549369e-02
# GAMMA= 5.943461e-05
OMEGA= 1.589966e+00
k_param=1.201872e+01
least_L_with_params(zeta=ZETA,delta=DELTA,omega=OMEGA,k=k_param,c0=1+1e-30)


277.70319734027

In [39]:
def objective(params):
    # Unpack parameters
    # zeta, epsilon, delta, gamma, omega, k = params
    zeta, delta, omega, k,tau = params

    # Call your existing bisection-based function
    # L = least_L_with_params(zeta=zeta, epsilon=epsilon,delta=delta, gamma=gamma, omega=omega, k=k)
    L = least_L_with_params(zeta=zeta, delta=delta, omega=omega,k=k,tau=tau,epsilon=0.25)

    # If the parameters are invalid, least_L_with_params returns 1e12.
    # This acts as a "penalty" that pushes the optimizer away from bad zones.
    return L


ZETA= 8.00000008622148817139e-01
DELTA = 4.166667249376546e-02
OMEGA= 1.596862190424492
k_param=12.020234190426336
TAU=0.500000587684954
# Bounds for (zeta, epsilon, gamma, omega, k)
# Using small buffers (1e-6) to avoid exact boundary issues
bounds = [
    (0.8, 0.9999),  # zeta: 0.8 < zeta < 1
    #(0.0001, 0.25),  # epsilon: typically small positive
    (0.0001,0.4), #delta: small positive
    # (1e-12, 0.1),  # gamma: 0 < gamma < epsilon**4 / 4
    (1.0001, 10.0),  # omega: typically > 1
    (1.0001, 50.0),  # k: k > 12
    (0.4,0.99999) # tau: 0.5=< tau < 1
]


# Initial guess (starting point for the optimizer)
x0 = [ZETA, DELTA, OMEGA, k_param,TAU]
# x0 = [ZETA, EPSILON,DELTA, GAMMA, OMEGA, k_param]

# We use COBYLA or Nelder-Mead because your function is likely not differentiable
res = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    bounds=bounds,
    options={'maxiter': 5000, 'disp': True}
)

if res.success:
    # optimized_zeta, optimized_delta, optimized_gamma, optimized_omega, optimized_k = res.x
    optimized_zeta, optimized_delta, optimized_omega, optimized_k,optimized_tau = res.x
    my_result = res.fun
    # optimized_zeta, optimized_epsilon,optimized_delta, optimized_gamma, optimized_omega, optimized_k = res.x
    print(f"Minimum L found: {my_result:4f}")
    if my_result < 400:
        print(f"Value of n : {np.exp(res.fun):.6e}")

    print(f"ZETA= {optimized_zeta:.20e}")
    # print(f"EPSILON= {optimized_epsilon:.6e}")
    print(f"DELTA = {optimized_delta:.15e}" )
    # print(f"GAMMA= {optimized_gamma:.15e}")
    print(f"OMEGA= {optimized_omega:.15f}")
    print(f"k_param={optimized_k:.15f}")
    print(f"TAU={optimized_tau:.15f}")


else:
    print("Optimization failed to converge.")

Optimization terminated successfully.
         Current function value: 256.925032
         Iterations: 167
         Function evaluations: 320
Minimum L found: 256.925032
Value of n : 3.811742e+111
ZETA= 8.00000008622148817139e-01
DELTA = 4.166667249376546e-02
OMEGA= 1.596862190424492
k_param=12.020234190426336
TAU=0.500000587684954


In [40]:
ZETA= 8.00000008622148817139e-01
DELTA = 4.166667249376546e-02
OMEGA= 1.596862190424492
k_param=12.020234190426336
TAU=0.500000587684954



least_L_with_params(ZETA,DELTA,OMEGA,k_param,TAU)

#best value found until here 256.92503165391827

256.92503165391827